# Forma Fechada para o Problema X21 — demonstração computacional

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x21_formas_fechadas.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*  
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Objetivo

Ilustrar numericamente a vantagem da forma fechada na solução do Problema X21B10T0.

A Eq. (2.51) decompõe $\Theta(x,t)$ em dois somatórios:

$$\Theta(x,t) = \underbrace{\frac{2\alpha}{L}\frac{q''_0}{k}\sum_{m=1}^{M}\frac{\cos(\beta_m x/L)}{A_m}}_{\text{série estacionária (convergência lenta)}} - \underbrace{\frac{2\alpha}{L}\frac{q''_0}{k}\sum_{m=1}^{M}\frac{e^{-A_m t}\cos(\beta_m x/L)}{A_m}}_{\text{série transiente (convergência rápida)}}$$

Substituindo a série estacionária pela forma fechada exata:

$$\sum_{m=1}^{\infty}\frac{\cos(\beta_m x/L)}{\beta_m^2} = \frac{1}{2}\left(1 - \frac{x}{L}\right)$$

obtém-se a Eq. (2.52), que converge com pouquíssimos termos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ── Parâmetros físicos ────────────────────────────────────────────────────────
L     = 66e-4        # espessura da placa [m]
alpha = 1.93e-7      # difusividade térmica [m²/s]
k     = 0.81         # condutividade térmica [W/(m·K)]
q0    = 1e3          # fluxo de calor em x=0 [W/m²]
T1    = 40.0         # temperatura inicial e em x=L [°C]

M_max = 500
m_arr = np.arange(1, M_max + 1)
beta  = np.pi * (m_arr - 0.5)
A     = alpha * beta**2 / L**2

In [ ]:
# ── Painel esquerdo: convergência do somatório estacionário ───────────────────
x_norm = np.linspace(0, 1, 400)
M_vals = [1, 2, 5, 20]
cores     = ['#D55E00', '#0072B2', '#009E73', '#CC79A7']
tracados  = ['--', '-.', ':', (0, (5, 1))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for M_val, cor, ls in zip(M_vals, cores, tracados):
    serie = np.sum(
        np.cos(np.outer(beta[:M_val], x_norm)) / beta[:M_val, None]**2, axis=0
    )
    ax1.plot(x_norm, serie, color=cor, linestyle=ls, linewidth=1.4, label=f'M = {M_val}')

ax1.plot(x_norm, 0.5*(1-x_norm), 'k-', linewidth=1.5, label='Forma fechada')
ax1.set_xlabel('x/L');  ax1.set_ylabel('Soma')
ax1.set_title('Convergência da série estacionária')
ax1.legend();  ax1.grid(True, linestyle='--', alpha=0.5)

# ── Painel direito: T(x=0, t) ─────────────────────────────────────────────────
t = np.linspace(0, 500, 600)
x = 0.0

def T_serie(M_val, t_vec):
    bm = beta[:M_val]; Am = A[:M_val]; cm = np.cos(bm*x/L)
    ss = (2*alpha/L)*(q0/k)*np.sum(cm/Am)
    tr = (2*alpha/L)*(q0/k)*np.sum((cm[:,None]*np.exp(-np.outer(Am,t_vec)))/Am[:,None], axis=0)
    return T1 + ss - tr

def T_fechada(M_val, t_vec):
    bm = beta[:M_val]; Am = A[:M_val]; cm = np.cos(bm*x/L)
    tr = (2*alpha/L)*(q0/k)*np.sum((cm[:,None]*np.exp(-np.outer(Am,t_vec)))/Am[:,None], axis=0)
    return T1 + (q0/k)*(L-x) - tr

ax2.plot(t, T_fechada(200, t), 'k-', linewidth=1.5, label='Referência (M=200)')
for M_val, cor, ls in zip([1, 2, 5], cores[:3], tracados[:3]):
    ax2.plot(t, T_serie(M_val, t), color=cor, linestyle=ls, linewidth=1.4, label=f'Eq.(2.51), M={M_val}')
ax2.plot(t, T_fechada(1, t), color=cores[3], linestyle=tracados[3], linewidth=1.4, label='Eq.(2.52), M=1')

ax2.set_xlabel('Tempo (s)');  ax2.set_ylabel('T(0, t) (°C)')
ax2.set_title('T(x=0, t): convergência em número de termos')
ax2.legend();  ax2.grid(True, linestyle='--', alpha=0.5)

fig.tight_layout()
plt.show()